# Baseline: Modelo Google no Itajaí-Açu

Este notebook realiza duas coisas:
1. **Treina um modelo local** usando o OpenHydroNet com nossos dados do Itajaí-Açu
2. **Calcula o baseline** (NSE, KGE) nos três eventos históricos críticos

---

## NSE e KGE: Por que são as métricas padrão em hidrologia?

Esta é uma das perguntas mais importantes para entender o campo. A resposta
revela o que "boa previsão" significa em hidrologia operacional.

### O problema com RMSE e MAE em hidrologia

Imagine duas bacias:
- **Bacia A** (Itajaí-Açu): média de 500 m³/s, picos de 8.000 m³/s
- **Bacia B** (Tijucas/SC): média de 50 m³/s, picos de 800 m³/s

Um modelo com RMSE = 200 m³/s é excelente para a Bacia A (4% da média)
mas catastrófico para a Bacia B (400% da média). RMSE e MAE são
**dependentes de escala** — não permitem comparar modelos entre bacias
ou contra a literatura.

### NSE — Nash-Sutcliffe Efficiency (1970)

```
           Σ(Q_sim - Q_obs)²
NSE = 1 − ─────────────────────
           Σ(Q_obs - mean(Q_obs))²
```

É o R² entre observado e simulado, mas com uma interpretação física:

| NSE | Significado |
|-----|-------------|
| 1.0 | Perfeito |
| 0.0 | Equivalente a prever sempre a média histórica |
| < 0 | Pior do que prever sempre a média |

**O denominador é o benchmark mais simples possível:** um modelo que
responde "a vazão de amanhã será a média histórica" tem NSE = 0.
NSE = 0.7 significa que seu modelo explica 70% da variância não explicada
pelo benchmark de climatologia.

**Limitação crítica do NSE:** o erro quadrático amplifica picos de cheia.
Um modelo pode ter NSE = 0.8 sendo excelente em dias normais e terrível
nas cheias — exatamente o caso mais importante. O NSE é "viciado" em
acertar a vazão mediana, não os extremos.

### KGE — Kling-Gupta Efficiency (2009)

Criado especificamente para resolver o problema do NSE. Decompõe o erro em três componentes:

```
KGE = 1 − √[(r−1)² + (α−1)² + (β−1)²]

Onde:
  r = correlação de Pearson  ← acerta o TIMING dos picos?
  α = std_sim / std_obs      ← acerta a AMPLITUDE das variações?
  β = mean_sim / mean_obs    ← acerta o VOLUME total?
```

| KGE | Significado |
|-----|-------------|
| 1.0 | Perfeito (r=1, α=1, β=1) |
| −0.41 | Equivalente ao benchmark de climatologia \* |
| < −0.41 | Pior do que prever sempre a média |

\* Knoben et al. (2019): o equivalente de NSE=0 para o KGE é −0.41, não 0!

**Por que KGE é superior para cheias?**
O componente α penaliza modelos que acertam a média mas subpreveem a
variabilidade (α < 1). Um modelo que "aplaina" os picos de cheia terá
α << 1 e portanto KGE baixo, mesmo que o NSE seja razoável.

### Qual usar?

A prática atual em hidrologia usa **ambas**:
- NSE para comparação com a literatura histórica (retrocompatibilidade)
- KGE como métrica primária de otimização e avaliação
- **FHV** (peak flow bias) como métrica especializada para eventos de cheia

O OpenHydroNet otimiza KGE implicitamente via NSELoss/cmalloss e reporta ambas.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys, os, subprocess
sys.path.insert(0, '../../')
# Vendor path para googlehydrology (instalado com pip install -e vendor/flood-forecasting)
# Se estiver usando o venv do projeto: source .venv/bin/activate
sys.path.insert(0, str(__import__('pathlib').Path('../..') / 'vendor' / 'flood-forecasting'))

from googlehydrology.evaluation import metrics as gh_metrics

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')

ROOT         = Path('../..')
CARAVAN_ROOT = ROOT / 'data/processed/Caravan-nc'
CONFIG_DIR   = ROOT / 'configs/training'
MODEL_DIR    = ROOT / 'models/experiments'
FIGDIR       = ROOT / 'reports/figures'
FIGDIR.mkdir(exist_ok=True)

CONFIG_FILE  = CONFIG_DIR / 'mef_lstm_itajai.yml'
BASIN_ID     = 'itajai_83500000'

FLOOD_EVENTS = {
    '1983-11-09': ('Nov/1983', '17.17m'),
    '2008-11-23': ('Nov/2008', '11.78m'),
    '2011-09-08': ('Set/2011', '10.14m'),
}

print('Ambiente configurado.')
print(f'Config: {CONFIG_FILE}')

## 1. Verificações Pré-Treinamento

Antes de iniciar o treinamento, verificamos que todos os pré-requisitos
estão satisfeitos. Falhar aqui é muito melhor do que falhar em epoch 28.

In [ ]:
import yaml
from src.data.caravan_formatter import validate_caravan_structure

checks_ok = True

# 1. Caravan dataset
caravan_checks = validate_caravan_structure(CARAVAN_ROOT)
if not all(caravan_checks.values()):
    print('ERRO: Caravan incompleto. Rode primeiro: notebooks/02_preprocessing/01_build_caravan_dataset.ipynb')
    checks_ok = False
else:
    print('✓ Caravan dataset OK')

# 2. Arquivo de lista de bacias
basins_file = CONFIG_DIR / 'itajai_basins.txt'
if not basins_file.exists():
    print('ERRO: itajai_basins.txt não encontrado')
    checks_ok = False
else:
    basins = [l.strip() for l in basins_file.read_text().splitlines() if l.strip()]
    print(f'✓ Lista de bacias: {basins}')

# 3. Config YAML
if not CONFIG_FILE.exists():
    print(f'ERRO: config não encontrado em {CONFIG_FILE}')
    checks_ok = False
else:
    with open(CONFIG_FILE) as f:
        cfg = yaml.safe_load(f)
    print(f'✓ Config carregado: {cfg["experiment_name"]}')
    print(f'  modelo:      {cfg["model"]}')
    print(f'  head:        {cfg["head"]}')
    print(f'  seq_length:  {cfg["seq_length"]} dias')
    print(f'  hidden_size: {cfg["hidden_size"]}')
    print(f'  epochs:      {cfg["epochs"]}')
    print(f'  device:      {cfg["device"]}')

# 4. OpenHydroNet instalado?
try:
    import googlehydrology
    print(f'✓ googlehydrology importado')
except ImportError:
    print('ERRO: googlehydrology não instalado.')
    print('  Execute: pip install -e upstream/')
    print('  (clonar primeiro: git clone https://github.com/google-research/flood-forecasting.git upstream/)')
    checks_ok = False

print(f'\n{"✓ Pronto para treinar" if checks_ok else "✗ Corrija os erros acima antes de continuar"}')

## 2. Treinamento

O OpenHydroNet é controlado por linha de comando: `run train --config-file=<path>`.

### Como funciona internamente

1. **Carregamento do dataset**: lê os arquivos NetCDF do Caravan-nc, cria
   janelas deslizantes de `seq_length` dias e as empilha em batches.
   Cada amostra de treino é uma tupla `(hindcast_sequence, forecast_sequence, target)`.

2. **Forward pass**: o MEF-LSTM processa hindcast e forecast em LSTMs separados;
   os estados finais são combinados pelo head de regressão para produzir
   as `lead_time` previsões futuras.

3. **Loss**: MSE calculado apenas nos últimos `predict_last_n=8` dias da sequência
   (os dias de forecast). O gradiente não flui para o período histórico.

4. **Validação**: ao final de cada epoch, calcula NSE e KGE no período de validação.

5. **Saída**: pesos em `models/experiments/{experiment_name}_{timestamp}/`;
   resultados de teste em `test/model_epoch{N}/test_results.zarr`.

In [ ]:
# Construir comando de treinamento
# O script 'run' é instalado pelo setup.py do googlehydrology como entry point.
train_cmd = f"run train --config-file={CONFIG_FILE.resolve()}"
infer_cmd_template = "run infer --run-dir=<MODEL_RUN_DIR>"

print('Comandos para executar no terminal:')
print(f'  conda activate blumenau-flood')
print(f'  cd {ROOT.resolve()}')
print(f'  {train_cmd}')
print()
print('Após completar o treino, substituir <MODEL_RUN_DIR> pelo diretório criado:')
print(f'  {infer_cmd_template}')
print()
print('O treinamento criará um diretório em:')
print(f'  {MODEL_DIR.resolve()}/{cfg["experiment_name"]}_YYYYMMDD_HHMMSS/')

# ── OPCIONAL: executar diretamente neste notebook ─────────────────────────────
# Descomente para treinar inline (bloqueia o notebook por ~10-30 min)
#
# import subprocess
# result = subprocess.run(train_cmd.split(), capture_output=True, text=True,
#                         cwd=str(ROOT.resolve()))
# print(result.stdout[-3000:] if result.stdout else '')
# if result.returncode != 0:
#     print('STDERR:', result.stderr[-1000:])

## 3. Carregando Resultados

Após o treinamento, o framework salva as predições em formato Zarr.
Vamos carregar o run mais recente automaticamente.

In [ ]:
import glob, re

def find_latest_run(model_dir: Path, experiment_name: str) -> Path | None:
    """Encontra o diretório do run mais recente para o experimento."""
    pattern = str(model_dir / f"{experiment_name}_*")
    candidates = sorted(glob.glob(pattern))
    return Path(candidates[-1]) if candidates else None

def load_test_results(run_dir: Path):
    """
    Carrega os resultados de teste do epoch mais recente.
    
    O framework salva em: test/model_epoch{N}/test_results.zarr
    Variáveis: streamflow_sim (predito), streamflow_obs (observado)
    Dimensões: basin × date × time_step (time_step = lead time em dias)
    """
    zarr_files = sorted(glob.glob(
        str(run_dir / 'test/model_epoch*/test_results.zarr')
    ))
    if not zarr_files:
        return None, None
    
    # Epoch mais recente
    def epoch_num(p):
        m = re.search(r'model_epoch(\d+)', p)
        return int(m.group(1)) if m else 0
    
    latest = max(zarr_files, key=epoch_num)
    epoch = epoch_num(latest)
    ds = xr.open_zarr(latest, consolidated=False)
    return ds, epoch


# Localizar run
run_dir = find_latest_run(MODEL_DIR, cfg['experiment_name'])

if run_dir is None:
    print(f'Nenhum run encontrado em {MODEL_DIR}.')
    print('Execute o treinamento na seção 2 antes de continuar.')
    ds_results = None
else:
    print(f'Run encontrado: {run_dir.name}')
    ds_results, epoch = load_test_results(run_dir)
    if ds_results is None:
        print('Resultados de teste não encontrados. Execute: run infer --run-dir=...')
    else:
        print(f'Resultados carregados (epoch {epoch})')
        print(ds_results)

## 4. NSE e KGE — Cálculo por Lead Time

O modelo gera previsões para lead times de 1 a 7 dias.
As métricas devem degradar com o horizonte — é esperado e fisicamente correto.

**Interpretação por lead time:**
- **Lead 1:** modelo "quase persiste" — a vazão de amanhã é parecida com a de hoje.
  NSE muito alto aqui não necessariamente indica modelo bom.
- **Lead 3–5:** teste real de capacidade preditiva. A memória do LSTM começa a
  superar a simples persistência.
- **Lead 7:** horizonte máximo. NSE > 0.5 aqui é um resultado forte.

In [ ]:
def compute_metrics_all_lead_times(ds: xr.Dataset, basin_id: str) -> pd.DataFrame:
    """
    Calcula métricas por lead time usando googlehydrology.evaluation.metrics.
    NSE, KGE e FHV são calculados pelo framework — mesma implementação usada
    na avaliação operacional do Google, sem reimplementação local.
    """
    results = []
    lead_times = ds['time_step'].values if 'time_step' in ds.dims else [0]

    for lt in lead_times:
        if 'time_step' in ds.dims:
            sim_da = ds['streamflow_sim'].sel(basin=basin_id, time_step=lt)
        else:
            sim_da = ds['streamflow_sim'].sel(basin=basin_id)

        obs_da = ds['streamflow_obs'].sel(basin=basin_id)

        # Alinhar datas (obs pode ter dimensão diferente de sim)
        common_dates = np.intersect1d(sim_da['date'].values, obs_da['date'].values)
        if len(common_dates) < 10:
            continue
        sim_da = sim_da.sel(date=common_dates)
        obs_da = obs_da.sel(date=common_dates)

        try:
            calc = gh_metrics.calculate_metrics(
                obs=obs_da,
                sim=sim_da,
                metrics=['NSE', 'KGE', 'FHV', 'Alpha-NSE', 'Beta-KGE', 'Pearson-r'],
                resolution='1D',
                datetime_coord='date',
            )
        except Exception as e:
            print(f'Lead {lt}: erro ao calcular métricas — {e}')
            continue

        results.append({
            'lead_time': int(lt),
            'NSE':      round(calc.get('NSE', float('nan')), 4),
            'KGE':      round(calc.get('KGE', float('nan')), 4),
            'FHV':      round(calc.get('FHV', float('nan')), 1),
            'alpha':    round(calc.get('Alpha-NSE', float('nan')), 4),
            'beta':     round(calc.get('Beta-KGE', float('nan')), 4),
            'r':        round(calc.get('Pearson-r', float('nan')), 4),
        })

    return pd.DataFrame(results).set_index('lead_time')


if ds_results is not None:
    metrics_df = compute_metrics_all_lead_times(ds_results, BASIN_ID)

    print('=== MÉTRICAS POR LEAD TIME ===')
    print(metrics_df.to_string())
    print()
    print('Interpretação KGE:')
    for lt, row in metrics_df.iterrows():
        kge = row['KGE']
        if kge > 0.7:
            verdict = 'Bom (operacional)'
        elif kge > 0.5:
            verdict = 'Satisfatório'
        elif kge > -0.41:
            verdict = 'Fraco (melhor que climatologia)'
        else:
            verdict = 'Pior que climatologia'
        print(f'  Lead {lt}: KGE={kge:.3f} — {verdict}')

    print()
    print('FHV (bias nos 2% maiores eventos):')
    print('  Positivo = modelo superestima cheias, Negativo = subestima')
    for lt, row in metrics_df.iterrows():
        print(f'  Lead {lt}: FHV = {row["FHV"]:+.1f}%')
else:
    print('Nenhum resultado disponível ainda. Execute o treinamento primeiro.')


## 5. Plot: Métricas × Lead Time

A degradação das métricas com o horizonte revela o comportamento do modelo:
- **Queda abrupta do NSE no lead 1→2:** o modelo não aprendeu padrões de
  médio prazo — depende demais da persistência
- **KGE mais estável que NSE:** a decomposição em r, α, β distribui o sinal;
  o modelo pode degradar em timing (r) mas manter volume (β)

In [ ]:
if ds_results is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    ax = axes[0]
    ax.plot(metrics_df.index, metrics_df['NSE'], marker='o', color='steelblue', label='NSE')
    ax.plot(metrics_df.index, metrics_df['KGE'], marker='s', color='seagreen', label='KGE')
    ax.axhline(0.7, color='gray', ls='--', lw=1, label='Alvo (0.70)')
    ax.axhline(0.0, color='k', ls=':', lw=0.8)
    ax.set_xlabel('Lead Time (dias)')
    ax.set_ylabel('Score')
    ax.set_title('NSE e KGE vs. Lead Time')
    ax.legend()
    ax.set_ylim(-0.5, 1.05)
    ax.set_xticks(metrics_df.index)
    
    ax2 = axes[1]
    ax2.plot(metrics_df.index, metrics_df['r'],     marker='o', label='r (timing)')
    ax2.plot(metrics_df.index, metrics_df['alpha'], marker='s', label='α (amplitude)')
    ax2.plot(metrics_df.index, metrics_df['beta'],  marker='^', label='β (volume)')
    ax2.axhline(1.0, color='k', ls='--', lw=0.8)
    ax2.set_xlabel('Lead Time (dias)')
    ax2.set_ylabel('Componente')
    ax2.set_title('Decomposição do KGE por Lead Time')
    ax2.legend(fontsize=9)
    ax2.set_xticks(metrics_df.index)
    
    ax3 = axes[2]
    colors = ['salmon' if v > 0 else 'steelblue' for v in metrics_df['FHV']]
    ax3.bar(metrics_df.index, metrics_df['FHV'], color=colors, alpha=0.8)
    ax3.axhline(0, color='k', lw=0.8)
    ax3.axhspan(-20, 20, alpha=0.08, color='green')
    ax3.set_xlabel('Lead Time (dias)')
    ax3.set_ylabel('FHV (%)')
    ax3.set_title('Bias nos Picos de Cheia (FHV)')
    ax3.set_xticks(metrics_df.index)
    ax3.text(0.98, 0.98, 'Verde: zona aceitável (±20%)',
             transform=ax3.transAxes, ha='right', va='top', fontsize=8)
    
    fig.suptitle(f'Baseline — Itajaí-Açu | {cfg["experiment_name"]}', fontsize=12)
    fig.tight_layout()
    fig.savefig(FIGDIR / '18_baseline_metrics_lead_time.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Resultados não disponíveis.')

## 6. Hidrogramas nos Eventos Históricos

Métricas agregadas escondem comportamento nos extremos.
Este plot revela **onde exatamente o modelo erra** durante as maiores cheias.

Padrões de falha comuns:
- **Timing**: pico previsto 1-2 dias cedo ou tarde
- **Magnitude**: pico previsto muito abaixo do real (problema crítico para alerta)
- **Recessão**: modelo superestima a vazão durante a descida do hidrograma
- **Pré-evento**: modelo não detecta o rápido aumento antes do pico

Cada um desses padrões tem um diagnóstico diferente:
- Timing errado → ajustar `seq_length` ou features de precipitação
- Pico subestimado → FHV muito negativo; considerar NSELoss ou peso nos extremos
- Recessão errada → `hidden_size` pequeno demais para capturar memória de baseflow

In [ ]:
def extract_event_window(
    ds: xr.Dataset,
    basin_id: str,
    peak_date: str,
    lead_time: int = 1,
    window_days: int = 30,
) -> pd.DataFrame | None:
    """
    Extrai observado + simulado em torno de um evento, para um lead time específico.
    """
    peak = pd.Timestamp(peak_date)
    t0   = peak - pd.Timedelta(days=window_days)
    t1   = peak + pd.Timedelta(days=window_days)
    
    try:
        obs = ds['streamflow_obs'].sel(basin=basin_id).to_series()
        if 'time_step' in ds.dims:
            sim = ds['streamflow_sim'].sel(basin=basin_id, time_step=lead_time).to_series()
        else:
            sim = ds['streamflow_sim'].sel(basin=basin_id).to_series()
        
        df = pd.DataFrame({'obs': obs, 'sim': sim})
        df.index = pd.to_datetime(df.index)
        return df.loc[t0:t1]
    except Exception as e:
        print(f'Erro ao extrair evento {peak_date}: {e}')
        return None


if ds_results is not None:
    LEAD_TIMES_TO_PLOT = [1, 3, 7]
    WINDOW = 25

    fig, axes = plt.subplots(
        len(FLOOD_EVENTS), len(LEAD_TIMES_TO_PLOT),
        figsize=(5 * len(LEAD_TIMES_TO_PLOT), 4.5 * len(FLOOD_EVENTS)),
        squeeze=False
    )

    for row, (date_str, (label, cota_str)) in enumerate(FLOOD_EVENTS.items()):
        for col, lt in enumerate(LEAD_TIMES_TO_PLOT):
            ax = axes[row][col]
            evt = extract_event_window(ds_results, BASIN_ID, date_str, lead_time=lt, window_days=WINDOW)

            if evt is None or evt.dropna().empty:
                ax.text(0.5, 0.5, 'Dados\nindisponíveis', ha='center', va='center',
                        transform=ax.transAxes)
                continue

            ax.plot(evt.index, evt['obs'], color='k', lw=1.5, label='Observado')
            ax.plot(evt.index, evt['sim'], color='steelblue', lw=1.5, ls='--',
                    label=f'Previsto (L{lt})')

            # Pico observado
            peak_dt = evt['obs'].idxmax()
            peak_v  = evt['obs'].max()
            ax.scatter([peak_dt], [peak_v], c='crimson', s=50, zorder=6)
            ax.axvline(pd.Timestamp(date_str), color='crimson', lw=1, ls=':', alpha=0.5)

            # Métricas do evento
            if lt in metrics_df.index:
                nse_evt = metrics_df.loc[lt, 'NSE']
                kge_evt = metrics_df.loc[lt, 'KGE']
                ax.text(0.02, 0.97,
                        f'NSE={nse_evt:.2f}\nKGE={kge_evt:.2f}',
                        transform=ax.transAxes, va='top', fontsize=8,
                        bbox=dict(facecolor='white', alpha=0.8))

            if col == 0:
                ax.set_title(f'{label} (cota {cota_str})', fontsize=10)
                ax.set_ylabel('Vazão (m³/s)')
            else:
                ax.set_title(f'Lead {lt} dias', fontsize=10)

            ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
            ax.legend(fontsize=7)

    fig.suptitle('Hidrogramas nos Eventos Históricos — Previsto vs. Observado',
                 fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig(FIGDIR / '19_baseline_hydrographs.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Execute o treinamento e inferência primeiro.')

## 7. Flow Duration Curve: Toda a Distribuição

A FDC (Flow Duration Curve) mostra o desempenho do modelo em **toda a distribuição**,
não apenas nos eventos extremos.

Gaps entre as curvas revelam:
- **Cabeça (Q > P10):** bias nos picos de cheia — coincide com FHV
- **Meio (P20–P70):** regime médio — onde NSE é mais sensível
- **Cauda (Q < P70):** estiagens — onde FLV mede o desempenho

In [ ]:
if ds_results is not None:
    lead_for_fdc = 1  # mostrar FDC para lead 1 dia (melhor cenário)
    
    obs_all = ds_results['streamflow_obs'].sel(basin=BASIN_ID).values.flatten()
    if 'time_step' in ds_results.dims:
        sim_all = ds_results['streamflow_sim'].sel(basin=BASIN_ID, time_step=lead_for_fdc).values
    else:
        sim_all = ds_results['streamflow_sim'].sel(basin=BASIN_ID).values.flatten()
    
    mask = (~np.isnan(obs_all)) & (~np.isnan(sim_all[:len(obs_all)]))
    obs_v = obs_all[mask]
    sim_v = sim_all[:len(obs_all)][mask]
    
    # Exceedance probabilities
    ep_obs = np.arange(1, len(obs_v)+1) / (len(obs_v)+1)
    ep_sim = np.arange(1, len(sim_v)+1) / (len(sim_v)+1)
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    # FDC em escala log
    axes[0].semilogy(ep_obs * 100, np.sort(obs_v)[::-1],
                     color='k', lw=2, label='Observado')
    axes[0].semilogy(ep_sim * 100, np.sort(sim_v)[::-1],
                     color='steelblue', lw=1.5, ls='--', label=f'Simulado (Lead {lead_for_fdc}d)')
    axes[0].set_xlabel('Probabilidade de Excedência (%)')
    axes[0].set_ylabel('Vazão (m³/s) — escala log')
    axes[0].set_title('Flow Duration Curve')
    axes[0].legend()
    axes[0].axvline(2, color='salmon', lw=1, ls=':', label='Região FHV (2%)')
    axes[0].axvline(70, color='gray', lw=1, ls=':')
    
    # Scatter obs vs. sim (escala log)
    lim_max = max(obs_v.max(), sim_v.max()) * 1.05
    axes[1].scatter(obs_v, sim_v, s=2, alpha=0.3, color='steelblue')
    axes[1].plot([0, lim_max], [0, lim_max], 'k--', lw=1, label='1:1')
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_xlabel('Observado (m³/s)')
    axes[1].set_ylabel('Simulado (m³/s)')
    axes[1].set_title(f'Scatter Obs vs. Sim — Lead {lead_for_fdc}d')
    
    fig.tight_layout()
    fig.savefig(FIGDIR / '20_baseline_fdc.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Execute o treinamento e inferência primeiro.')

## 8. Diagnóstico e Próximos Passos

Com base nas métricas e hidrogramas acima, preenchemos o diagnóstico do baseline.

In [ ]:
if ds_results is not None and not metrics_df.empty:
    lt1 = metrics_df.loc[1] if 1 in metrics_df.index else metrics_df.iloc[0]
    lt7 = metrics_df.loc[7] if 7 in metrics_df.index else metrics_df.iloc[-1]
    
    print('=' * 55)
    print('  DIAGNÓSTICO DO BASELINE')
    print('=' * 55)
    print(f'  NSE Lead-1:  {lt1["NSE"]:.3f}   KGE Lead-1:  {lt1["KGE"]:.3f}')
    print(f'  NSE Lead-7:  {lt7["NSE"]:.3f}   KGE Lead-7:  {lt7["KGE"]:.3f}')
    print(f'  FHV Lead-1:  {lt1["FHV"]:+.1f}%  FHV Lead-7:  {lt7["FHV"]:+.1f}%')
    print(f'  Decomposição Lead-1: r={lt1["r"]:.3f}, α={lt1["alpha"]:.3f}, β={lt1["beta"]:.3f}')
    print()
    
    # Diagnóstico automático
    issues = []
    if lt1['alpha'] < 0.8:
        issues.append('α < 0.8: modelo subestima variabilidade → considerar NSELoss ou aumentar hidden_size')
    if abs(lt1['beta'] - 1) > 0.2:
        issues.append(f'β = {lt1["beta"]:.2f}: bias sistemático de volume → verificar normalização')
    if lt1['r'] < 0.8:
        issues.append('r < 0.8: timing ruim → considerar aumentar seq_length ou adicionar features de precipitação')
    if abs(lt1['FHV']) > 30:
        issues.append(f'FHV = {lt1["FHV"]:+.0f}%: grande bias nos picos → modelo pode falhar em alertas de cheia')
    if lt1['KGE'] > 0.7:
        issues.append('KGE Lead-1 > 0.7: baseline sólido. Próximo passo: fine-tuning para melhorar Lead-7.')
    elif lt1['KGE'] > 0.5:
        issues.append('KGE Lead-1 entre 0.5–0.7: modelo funcional mas abaixo do alvo operacional de 0.7.')
    else:
        issues.append('KGE Lead-1 < 0.5: modelo abaixo do esperado. Verificar pipeline de dados antes de fine-tuning.')
    
    print('  Diagnóstico:')
    for issue in issues:
        print(f'  → {issue}')
    
    print()
    print('  Próximos passos sugeridos:')
    print('  1. notebooks/03_training/02_finetune.ipynb — fine-tune nos eventos extremos')
    print('  2. Adicionar ERA5-Land como features dinâmicas (temperatura, evapotranspiração)')
    print('  3. Extrair atributos HydroAtlas e ERA5 para melhorar o embedding estático')
    print('  4. Experimento de ablação: seq_length (14 vs 30 vs 60)')
    print('=' * 55)
else:
    print('Execute o treinamento e seções 3–5 antes do diagnóstico.')